<a href="https://colab.research.google.com/github/johanndebodastudent/ai-social-good-project/blob/main/Milestone_2_Anton_Bohlin.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Colab Link:https://colab.research.google.com/drive/1nLgbuBIeMZywiz6ZTsR57ts6BzgjcD_w?usp=sharing

Section 1: Problem and Population

Marcus is a 45-year-old resident of Santa Clara County unhoused near St. James Park after a construction injury. The specific failure point is that information regarding shelter amenities, specifically secure storage for his construction tools—is effectively "invisible" because it is buried in unstructured PDFs and scattered city websites. This project addresses SDG 11: Sustainable Cities and Communities.

M1 Update: Initially, the project focused on general directories; it has now shifted to a "Wellness-First" triage system that identifies personal property (tools) to ensure Marcus is met with social services rather than displacement-focused clean-up crews.

Section 2: Proposed System

Input: A photo of a site and a brief description of observed needs.

AI Processing:
* Image Analysis: Identifies signs of human residence and construction equipment.
* Structured Extraction: Formats messy data into a JSON record that flags for "Tool Storage Required".

Output: A prioritized "Vulnerability Report" sent to city dispatch.
* Real-World Action: The system automatically routes the case to Social Services for matching with a tool-safe shelter instead of a standard waste removal crew

Section 3: Project Code

In [ ]:
import google.generativeai as genai
from google.colab import userdata
import json
import time

genai.configure(api_key=userdata.get('GEMINI_API_KEY'))
print("Gemini initialized successfully.")

Gemini initialized successfully.


Component 1: Visual Asset Identification

In [ ]:
import PIL.Image

# Load local image asset for analysis
img = PIL.Image.open('marcus_site.jpg')

# Initialize Flash model for multimodal processing
model = genai.GenerativeModel('gemini-2.5-flash')

prompt = "Identify if this image contains construction tools or personal belongings (tents/bedding). Rate urgency for social services from 1-10."

# Execute content generation using image and triage prompt
response = model.generate_content([img, prompt])
print(response.text)

This image contains **construction tools** and mechanical repair tools.

It does **not** contain personal belongings (tents/bedding).

Urgency for social services: **1** (indicating no urgency, as this appears to be a well-equipped workshop or garage, not a scene of distress or need for social services).


This component demonstrates the AI's ability to 'see' Marcus's tools. Notably, the AI rated the urgency as a 1, which shows a limitation in its training regarding unhoused residents' livelihoods. My proposed system would include a logic override: if 'construction tools' are identified, the urgency is automatically elevated for human review."

Component 2: Resource Matching Schema

In [ ]:
# High-Precision Structured Extraction Logic for Marcus
schema_prompt = """
Return ONLY valid JSON with exactly these five fields:
{
  "location": "string",
  "primary_need": "string",
  "urgency": "LOW, MEDIUM, or HIGH",
  "has_construction_tools": "boolean",
  "recommended_resource": "string"
}
No explanation. JSON only.
"""

def extract_structured(message):
    # Utilizing the optimized 2.5 Flash model for high-speed triage.
    m = genai.GenerativeModel(model_name="gemini-2.5-flash", system_instruction=schema_prompt)
    response = m.generate_content(message)
    # This cleans the AI output to ensure it is valid JSON
    raw = response.text.strip().replace('```json', '').replace('```', '')
    return json.loads(raw)

# Test run for Marcus
marcus_msg = "I'm near St. James Park. I need a bed tonight but I have my expensive construction tools with me and I'm worried they will be stolen."
result = extract_structured(marcus_msg)
print(json.dumps(result, indent=2))

{
  "location": "St. James Park",
  "primary_need": "Shelter",
  "urgency": "HIGH",
  "has_construction_tools": true,
  "recommended_resource": "Contact a local homelessness services helpline or coordinated entry program, specifying your need for secure storage for valuable tools when inquiring about shelter availability."
}


This demonstrates how unstructured crisis reports are turned into searchable data points, allowing the city to filter for 'Tool Storage' needs specifically for Marcus.

Section 4: Edge Case Elicitation & Risk Assessment

In [ ]:
# Testing an ambiguous input: a backpack on a park bench
edge_case = "There is a single backpack and a hard hat sitting on a bench in St. James Park. No one is around."
edge_result = extract_structured(edge_case)
print(json.dumps(edge_result, indent=2))

{
  "location": "St. James Park",
  "primary_need": "Investigate unattended items",
  "urgency": "LOW",
  "has_construction_tools": false,
  "recommended_resource": "Park Ranger or Local Authorities (non-emergency)"
}


Prompt: Analysis of a backpack/hard hat on a bench.

AI Output: {
  "location": "St. James Park",
  "primary_need": "Lost and Found / Securing unattended property",
  "urgency": "LOW",
  "has_construction_tools": false,
  "recommended_resource": "Park Rangers or Local Police (non-emergency)"
}

Assessment: Near-miss. The system correctly identified the items as "unattended" but recommended a Park Ranger for investigation. In Marcus’s context, this is a near-miss; while the tools aren't being treated as trash, a ranger response could still lead to his livelihood being impounded while he is away from his site.